# Franka Pick-and-Place – Training Analysis

Visualise reward curves from TensorBoard logs and per-episode results from `evaluate.py`.

> **No Isaac Sim required.** Run with any standard Python kernel.

In [ ]:
import os
import json
import glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Project root (one level up from this notebook if moved, or same dir)
PROJECT_DIR = os.path.dirname(os.path.abspath('.'))
LOG_DIR     = os.path.join(os.path.dirname(os.path.abspath('analysis.ipynb')), 'logs')
MODEL_DIR   = os.path.join(os.path.dirname(os.path.abspath('analysis.ipynb')), 'model')

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print(f'Log dir  : {LOG_DIR}')
print(f'Model dir: {MODEL_DIR}')

## 1 – Reward curve from TensorBoard logs

Export from TensorBoard as CSV, or use `tensorboard --logdir logs/` and export manually.
Alternatively, the cell below reads the raw TFEvents files directly via `tensorflow`.

In [ ]:
# ── Option A: load a CSV exported from TensorBoard ────────────────────────
csv_files = sorted(glob.glob(os.path.join(LOG_DIR, '**', '*.csv'), recursive=True))

if csv_files:
    df = pd.read_csv(csv_files[0])
    print(f'Loaded: {csv_files[0]}')
    print(df.head())
else:
    print('No CSV files found in logs/. Export from TensorBoard or run evaluate.py --save-results.')
    df = None

In [ ]:
# ── Option B: read TFEvents files directly (requires tensorflow / tensorboard) ─
try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

    event_files = sorted(glob.glob(os.path.join(LOG_DIR, '**', 'events.out.*'), recursive=True))
    if not event_files:
        raise FileNotFoundError('No TFEvent files found in logs/')

    ea = EventAccumulator(os.path.dirname(event_files[0]))
    ea.Reload()

    scalar_tags = ea.Tags().get('scalars', [])
    print('Available scalar tags:', scalar_tags)

    reward_tag = next((t for t in scalar_tags if 'reward' in t.lower()), None)
    if reward_tag:
        events = ea.Scalars(reward_tag)
        df_tb = pd.DataFrame([(e.step, e.value) for e in events], columns=['step', 'reward'])
        print(f'\nLoaded {len(df_tb)} reward data points (tag: {reward_tag})')
    else:
        print('No reward scalar found. Available tags:', scalar_tags)
        df_tb = None

except Exception as exc:
    print(f'TFEvent loading skipped: {exc}')
    df_tb = None

In [ ]:
# ── Plot reward curve ─────────────────────────────────────────────────────────
plot_df = df_tb if (df_tb is not None and len(df_tb) > 0) else df

if plot_df is not None:
    step_col   = 'step'   if 'step'   in plot_df.columns else plot_df.columns[1]
    reward_col = 'reward' if 'reward' in plot_df.columns else plot_df.columns[2]

    # Smooth with rolling mean
    smoothed = plot_df[reward_col].rolling(window=10, min_periods=1).mean()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(plot_df[step_col], plot_df[reward_col], alpha=0.3, color='steelblue', label='Raw')
    ax.plot(plot_df[step_col], smoothed, color='steelblue', linewidth=2, label='Smoothed (w=10)')
    ax.set_xlabel('Timestep')
    ax.set_ylabel('Episode Reward')
    ax.set_title('PPO Training – Episode Reward')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No reward data to plot yet. Run training first.')

## 2 – Evaluation results

Generated by:
```bash
python.sh evaluate.py --save-results logs/eval_results.json
```

In [ ]:
eval_files = sorted(glob.glob(os.path.join(LOG_DIR, '**', '*.json'), recursive=True))

if eval_files:
    with open(eval_files[-1]) as f:  # most recent
        eval_data = json.load(f)
    eval_df = pd.DataFrame(eval_data['results'])
    print(f'Loaded: {eval_files[-1]}')
    print(eval_df.to_string(index=False))
else:
    print('No eval JSON found. Run: python.sh evaluate.py --save-results logs/eval_results.json')
    eval_df = None

In [ ]:
if eval_df is not None:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # Per-episode reward
    axes[0].bar(eval_df['episode'], eval_df['reward'], color='steelblue')
    axes[0].set_xlabel('Episode')
    axes[0].set_ylabel('Total Reward')
    axes[0].set_title('Reward per Episode')
    axes[0].grid(axis='y', alpha=0.3)

    # Touch / lift success
    n = len(eval_df)
    categories = ['Touch', 'Lift']
    rates = [eval_df['touched'].mean() * 100, eval_df['lifted'].mean() * 100]
    colors = ['#4CAF50' if r > 50 else '#F44336' for r in rates]
    axes[1].bar(categories, rates, color=colors)
    axes[1].set_ylim(0, 100)
    axes[1].set_ylabel('Success Rate (%)')
    axes[1].set_title(f'Success Rates  (n={n})')
    axes[1].grid(axis='y', alpha=0.3)
    for i, v in enumerate(rates):
        axes[1].text(i, v + 2, f'{v:.0f}%', ha='center', fontweight='bold')

    # Episode length
    axes[2].bar(eval_df['episode'], eval_df['steps'], color='darkorange')
    axes[2].set_xlabel('Episode')
    axes[2].set_ylabel('Steps')
    axes[2].set_title('Episode Length')
    axes[2].grid(axis='y', alpha=0.3)

    plt.suptitle('Evaluation Results', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No evaluation data to plot yet.')